# Dylan Wendling 
# Final project
# A 2D Artillery Game featuring dynamic terrain, maps, phasing projectiles, and a fully functional pause menu.

In [ ]:
# File: pyshock.py
# A 2D Artillery Game featuring destructible terrain, hot-seat multiplayer, and multiple weapons.

import pygame
import math
import random
import sys

# --- CONFIGURATION & CONSTANTS ---
WIDTH, HEIGHT = 800, 600
FPS = 60

# Colors
SKY_BLUE = (135, 206, 235)
CLOUD_COLOR = (255, 255, 255, 180)
MOUNTAIN_COLOR = (100, 110, 130)
TERRAIN_COLOR = (34, 139, 34)
CRATER_COLOR = (139, 69, 19)
P1_COLOR = (50, 50, 255)
P2_COLOR = (255, 50, 50)
UI_TEXT = (255, 255, 255)

# Physics
GRAVITY = 0.2
MAX_POWER = 25
EXPLOSION_RADIUS = 40
DIRT_BOMB_RADIUS = 50
MAX_HEALTH = 100
MOVE_SPEED = 2

class Background:
    def __init__(self):
        self.clouds = []
        self.mountains = []
        self.generate_mountains()
        self.generate_clouds()

    def generate_mountains(self):
        # Create a static distant mountain range
        self.mountains = [(0, HEIGHT)]
        for x in range(0, WIDTH + 100, 100):
            self.mountains.append((x, random.randint(HEIGHT // 3, HEIGHT // 2 + 50)))
        self.mountains.append((WIDTH, HEIGHT))

    def generate_clouds(self):
        # Procedural clouds: x, y, size, speed
        for _ in range(8):
            self.clouds.append([
                random.randint(0, WIDTH), 
                random.randint(20, 150), 
                random.randint(20, 50), 
                random.uniform(0.1, 0.4)
            ])

    def update_and_draw(self, surface):
        # Draw Mountains (Static Parallax Layer)
        pygame.draw.polygon(surface, MOUNTAIN_COLOR, self.mountains)
        
        # Draw and Update Clouds (Scrolling Parallax Layer)
        for cloud in self.clouds:
            cloud[0] += cloud[3] # Move cloud by speed
            if cloud[0] - cloud[2] > WIDTH:
                cloud[0] = -cloud[2] # Wrap around screen
            
            # Draw overlapping circles for a fluffy cloud look
            pygame.draw.circle(surface, CLOUD_COLOR, (int(cloud[0]), int(cloud[1])), int(cloud[2]))
            pygame.draw.circle(surface, CLOUD_COLOR, (int(cloud[0] + cloud[2]*0.8), int(cloud[1] + cloud[2]*0.2)), int(cloud[2]*0.8))
            pygame.draw.circle(surface, CLOUD_COLOR, (int(cloud[0] - cloud[2]*0.6), int(cloud[1] + cloud[2]*0.3)), int(cloud[2]*0.7))

class Terrain:
    def __init__(self, width, height):
        self.width = width
        self.height = height
        self.heightmap = []
        self.generate()

    def generate(self):
        self.heightmap = []
        base_height = self.height * 0.7
        offset1 = random.uniform(0, 2 * math.pi)
        offset2 = random.uniform(0, 2 * math.pi)
        for x in range(self.width):
            h = base_height + math.sin((x / 150) + offset1) * 60 + math.sin((x / 50) + offset2) * 20
            self.heightmap.append(int(h))

    def draw(self, surface):
        for x in range(self.width):
            y = self.heightmap[x]
            pygame.draw.line(surface, TERRAIN_COLOR, (x, y), (x, self.height), 1)
            pygame.draw.line(surface, CRATER_COLOR, (x, y), (x, y+5), 1)

    def modify_terrain(self, cx, cy, radius, is_dirt_bomb=False):
        cx, cy = int(cx), int(cy)
        for x in range(max(0, cx - radius), min(self.width, cx + radius)):
            dx = x - cx
            dy = math.sqrt(max(0, radius**2 - dx**2))
            
            if is_dirt_bomb:
                # Add a mound of dirt (raise the heightmap, which means lower the y-value)
                mound_top = cy - dy
                if self.heightmap[x] > mound_top:
                    self.heightmap[x] = int(mound_top)
            else:
                # Normal explosion: Carve a crater (lower the heightmap, which means raise the y-value)
                explosion_bottom = cy + dy
                if self.heightmap[x] < explosion_bottom:
                    self.heightmap[x] = int(explosion_bottom)

class Tank:
    def __init__(self, x, color, name):
        self.x = x
        self.y = 0
        self.color = color
        self.name = name
        self.health = MAX_HEALTH
        self.angle = 45 if x < WIDTH/2 else 135
        self.power = 15
        self.score = 0
        self.width = 20
        self.height = 10
        self.weapons = ["Standard", "Cluster", "Dirt Bomb"]
        self.weapon_idx = 0

    def get_current_weapon(self):
        return self.weapons[self.weapon_idx]

    def cycle_weapon(self):
        self.weapon_idx = (self.weapon_idx + 1) % len(self.weapons)

    def move(self, direction, terrain):
        # direction: -1 for left, 1 for right
        self.x += direction * MOVE_SPEED
        # Keep tank on screen boundaries
        self.x = max(self.width, min(WIDTH - self.width, self.x))
        self.update_position(terrain)

    def update_position(self, terrain):
        self.y = terrain.heightmap[int(self.x)]

    def draw(self, surface):
        if self.health <= 0: return
        rect = pygame.Rect(self.x - self.width//2, self.y - self.height, self.width, self.height)
        pygame.draw.rect(surface, self.color, rect)
        pygame.draw.circle(surface, self.color, (int(self.x), int(self.y - self.height)), self.width//2)

        rad = math.radians(self.angle)
        end_x = self.x + math.cos(rad) * 20
        end_y = self.y - self.height - math.sin(rad) * 20
        pygame.draw.line(surface, (0, 0, 0), (self.x, self.y - self.height), (end_x, end_y), 4)

class Projectile:
    def __init__(self, x, y, angle, power, wind, weapon_type):
        self.x = x
        self.y = y
        rad = math.radians(angle)
        self.vx = math.cos(rad) * power
        self.vy = -math.sin(rad) * power
        self.wind = wind
        self.weapon_type = weapon_type
        self.trail = []

    def update(self):
        self.trail.append((int(self.x), int(self.y)))
        if len(self.trail) > 15: self.trail.pop(0)

        self.vy += GRAVITY
        self.vx += self.wind
        self.x += self.vx
        self.y += self.vy

    def draw(self, surface):
        color = (139, 69, 19) if self.weapon_type == "Dirt Bomb" else (50, 50, 50)
        if len(self.trail) > 1:
            pygame.draw.lines(surface, (200, 200, 200), False, self.trail, 2)
        pygame.draw.circle(surface, color, (int(self.x), int(self.y)), 4)

class Game:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WIDTH, HEIGHT))
        pygame.display.set_caption("PyShock - Multiplayer Arsenal")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont("Arial", 16, bold=True)
        self.big_font = pygame.font.SysFont("Arial", 36, bold=True)
        
        self.background = Background()
        self.terrain = Terrain(WIDTH, HEIGHT)
        self.p1 = Tank(100, P1_COLOR, "Player 1")
        self.p2 = Tank(WIDTH - 100, P2_COLOR, "Player 2")
        self.wind = random.uniform(-0.1, 0.1)
        
        self.active_player = self.p1
        self.projectiles = []
        self.explosions = [] # Stores dicts: {'pos': (x,y), 'timer': 30, 'type': "Standard"}
        self.state = "AIMING" 
        
        self.p1.update_position(self.terrain)
        self.p2.update_position(self.terrain)

    def handle_input(self):
        keys = pygame.key.get_pressed()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
                
            if event.type == pygame.KEYDOWN:
                if self.state == "AIMING":
                    if event.key == pygame.K_SPACE:
                        self.fire_projectile()
                    if event.key == pygame.K_TAB:
                        self.active_player.cycle_weapon()
                elif self.state == "GAMEOVER" and event.key == pygame.K_RETURN:
                    self.reset_round()

        if self.state == "AIMING":
            # Movement
            if keys[pygame.K_a]: self.active_player.move(-1, self.terrain)
            if keys[pygame.K_d]: self.active_player.move(1, self.terrain)
            # Aiming & Power
            if keys[pygame.K_LEFT]: self.active_player.angle = min(180, self.active_player.angle + 1)
            if keys[pygame.K_RIGHT]: self.active_player.angle = max(0, self.active_player.angle - 1)
            if keys[pygame.K_UP]: self.active_player.power = min(MAX_POWER, self.active_player.power + 0.2)
            if keys[pygame.K_DOWN]: self.active_player.power = max(1, self.active_player.power - 0.2)

    def fire_projectile(self):
        start_x = self.active_player.x
        start_y = self.active_player.y - self.active_player.height
        weapon = self.active_player.get_current_weapon()
        
        if weapon == "Cluster":
            # Fire 3 projectiles at slightly different angles
            for offset in [-5, 0, 5]:
                p = Projectile(start_x, start_y, self.active_player.angle + offset, self.active_player.power, self.wind, weapon)
                self.projectiles.append(p)
        else:
            p = Projectile(start_x, start_y, self.active_player.angle, self.active_player.power, self.wind, weapon)
            self.projectiles.append(p)
            
        self.state = "FIRING"

    def process_projectiles(self):
        for p in self.projectiles[:]:
            p.update()
            
            # Remove if completely out of bounds laterally or bottom
            if p.x < -100 or p.x > WIDTH + 100 or p.y > HEIGHT + 100:
                self.projectiles.remove(p)
                continue

            px, py = int(p.x), int(p.y)
            if 0 <= px < WIDTH and py >= self.terrain.heightmap[px]:
                # Terrain Collision Hit!
                self.projectiles.remove(p)
                
                is_dirt = (p.weapon_type == "Dirt Bomb")
                radius = DIRT_BOMB_RADIUS if is_dirt else EXPLOSION_RADIUS
                
                self.terrain.modify_terrain(px, py, radius, is_dirt_bomb=is_dirt)
                self.explosions.append({'pos': (px, py), 'timer': 30, 'type': p.weapon_type})
                
                if not is_dirt:
                    self.damage_tanks(px, py, radius)

    def process_explosions(self):
        for exp in self.explosions[:]:
            exp['timer'] -= 1
            if exp['timer'] <= 0:
                self.explosions.remove(exp)

    def damage_tanks(self, cx, cy, radius):
        for tank in [self.p1, self.p2]:
            if tank.health <= 0: continue
            dist = math.hypot(tank.x - cx, tank.y - cy)
            if dist < radius + 10:
                damage = int((1 - (dist / (radius + 10))) * 40)
                tank.health -= max(0, damage)
                if tank.health <= 0:
                    tank.health = 0
                    self.state = "GAMEOVER"
                    if tank == self.p1: self.p2.score += 1
                    else: self.p1.score += 1

    def next_turn(self):
        self.wind = random.uniform(-0.1, 0.1)
        self.active_player = self.p2 if self.active_player == self.p1 else self.p1
        self.state = "AIMING"

    def reset_round(self):
        self.terrain.generate()
        for tank in [self.p1, self.p2]:
            tank.health = MAX_HEALTH
            tank.update_position(self.terrain)
        self.wind = random.uniform(-0.1, 0.1)
        self.state = "AIMING"
        self.active_player = self.p1

    def draw_ui(self):
        # Top Info Bar
        wind_dir = ">>" if self.wind > 0 else "<<"
        wind_text = self.font.render(f"Wind: {abs(self.wind*100):.1f} {wind_dir}", True, UI_TEXT)
        self.screen.blit(wind_text, (WIDTH//2 - 40, 10))
        
        turn_text = self.font.render(f"{self.active_player.name}'s Turn", True, self.active_player.color)
        self.screen.blit(turn_text, (WIDTH//2 - 50, 30))

        # P1 Stats
        p1_stats = self.font.render(f"HP: {self.p1.health} | Ang: {self.p1.angle} | Pow: {self.p1.power:.1f}", True, P1_COLOR)
        p1_wep = self.font.render(f"Wep: {self.p1.get_current_weapon()} | Score: {self.p1.score}", True, P1_COLOR)
        self.screen.blit(p1_stats, (10, 10))
        self.screen.blit(p1_wep, (10, 30))

        # P2 Stats
        p2_stats = self.font.render(f"HP: {self.p2.health} | Ang: {self.p2.angle} | Pow: {self.p2.power:.1f}", True, P2_COLOR)
        p2_wep = self.font.render(f"Wep: {self.p2.get_current_weapon()} | Score: {self.p2.score}", True, P2_COLOR)
        self.screen.blit(p2_stats, (WIDTH - 250, 10))
        self.screen.blit(p2_wep, (WIDTH - 250, 30))

        # Controls Help
        controls = self.font.render("Move: A/D | Aim: Arrows | Wep: TAB | Fire: SPACE", True, (200, 200, 200))
        self.screen.blit(controls, (WIDTH//2 - 170, HEIGHT - 25))

        if self.state == "GAMEOVER":
            winner = "Player 1" if self.p2.health <= 0 else "Player 2"
            msg = self.big_font.render(f"{winner} Wins! Press ENTER.", True, (255, 255, 0))
            self.screen.blit(msg, (WIDTH//2 - 180, HEIGHT//2))

    def update(self):
        if self.state == "AIMING":
            # Just keep tanks snapped to ground if dirt bomb hit them
            self.p1.update_position(self.terrain)
            self.p2.update_position(self.terrain)

        elif self.state == "FIRING":
            self.process_projectiles()
            self.process_explosions()
            # If all shots landed and explosions faded, pass turn (unless someone died)
            if len(self.projectiles) == 0 and len(self.explosions) == 0:
                if self.state != "GAMEOVER":
                    self.next_turn()

    def draw(self):
        # Draw Background Layers
        self.screen.fill(SKY_BLUE)
        self.background.update_and_draw(self.screen)
        
        # Draw Gameplay Layers
        self.terrain.draw(self.screen)
        self.p1.draw(self.screen)
        self.p2.draw(self.screen)

        for p in self.projectiles:
            p.draw(self.screen)

        for exp in self.explosions:
            color = (139, 69, 19) if exp['type'] == "Dirt Bomb" else (255, 100, 0)
            pygame.draw.circle(self.screen, color, exp['pos'], EXPLOSION_RADIUS)

        self.draw_ui()
        pygame.display.flip()

    def run(self):
        while True:
            self.handle_input()
            self.update()
            self.draw()
            self.clock.tick(FPS)

if __name__ == "__main__":
    game = Game()
    game.run()